# 119 — Evaluación y depuración de agentes

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Dos ejes complementarios:

- **Eval de resultado:** ¿el estado final satisface el objetivo? (predicado
  ejecutable: tests pasan, archivo válido). Es lo que importa al usuario; ignora el
  camino.
- **Eval de proceso:** ¿el CÓMO fue correcto? — tools pertinentes, argumentos
  válidos, permisos respetados, presupuesto razonable. Detecta éxitos por casualidad
  (✓ resultado, ✗ proceso: fallará pronto) y fallos por una decisión reparable.

La matriz 2×2 resultado×proceso es la primera herramienta diagnóstica. Métricas sobre
un conjunto de tareas reproducible: tasa de éxito (con su varianza), pass@k, costo
por éxito, pasos vs óptimo, violaciones. LLM-as-judge para criterios blandos — con
rúbrica y calibración contra humanos.

### 🔬 Depurar = leer trayectorias

Método: (1) reunir trayectorias FALLIDAS del eval; (2) localizar el **primer paso
divergente** (la primera decisión que un experto no tomaría — el fallo visible suele
ser síntoma posterior); (3) clasificar la causa raíz (E1 instrucciones, E2 selección
de tool, E3 argumentos, E4 interpretación de la observación, E5 planificación,
E6 parada, E7 entorno/tool); (4) CONTAR y arreglar la categoría dominante;
(5) re-ejecutar el eval completo — sin re-ejecución no hay evidencia de mejora ni
detección de regresiones.

El laboratorio `evaluation` entrega la matriz mínima: tp=3, fp=1, fn=1 →
precision = recall = 0,75, con la advertencia honesta de que 8 ejemplos no estiman
desempeño real.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Verifica las métricas a mano.** Ejecuta `run_lab("evaluation",
seed=119)`. Calcula a mano precision y recall desde tp/fp/fn y verifica contra el
resultado. Añade F1. Si el costo de un falso negativo fuera 10 veces el de un falso
positivo, ¿qué métrica vigilarías y por qué?

**Ejercicio 2 — Matriz 2×2.** Un eval de 12 tareas dio: 6 con resultado ✓ y proceso ✓;
2 con resultado ✓ pero pasos redundantes y un test borrado; 3 con resultado ✗ por
timeout de una API externa; 1 con resultado ✗ por plan erróneo desde el inicio.
Construye la matriz 2×2, calcula tasa de éxito ingenua y honesta, y asigna categoría
E1-E7 a cada grupo de fallos.

**Ejercicio 3 — Primer paso divergente.** Trayectoria fallida: [1] lee el bug report;
[2] busca en el módulo equivocado (el reporte menciona `auth/`, busca en `api/`);
[3] edita un archivo de `api/`; [4] los tests siguen fallando; [5] re-edita lo mismo;
[6] presupuesto agotado. Señala el primer paso divergente, la categoría E, y qué
intervención (prompt, tool description, plan) elegirías. ¿Por qué el paso 5 NO es la
causa?

**Ejercicio 4 — Detector de regresión.** Implementa `comparar_evals(antes, despues)`
que reciba dos listas de resultados por tarea (`{"id", "exito", "costo"}`) y reporte:
Δ tasa de éxito, tareas que pasaron de ✓ a ✗ (regresiones), de ✗ a ✓ (mejoras) y
Δ costo por éxito. Pruébalo con datos donde la tasa global MEJORA pero hay 2
regresiones — y explica por qué ese caso exige revisión antes de desplegar.

In [ ]:
# TODO: ejecuta run_lab("evaluation", seed=119)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: métricas a mano
result = run_lab("evaluation", seed=119)
r = result["result"]
tp, fp, fn = r["tp"], r["fp"], r["fn"]
precision = None  # tp / (tp + fp)
recall = None     # tp / (tp + fn)
f1 = None         # 2*p*r/(p+r)
# ¿qué métrica vigilas si FN cuesta 10x FP?


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: detector de regresión
def comparar_evals(antes, despues):
    # antes/despues: [{"id": ..., "exito": bool, "costo": float}, ...]
    # devuelve: delta_tasa, regresiones, mejoras, delta_costo_por_exito
    pass


## Reflexión

1. Un agente "arregló" el build borrando el test que fallaba: resultado ✓, proceso ✗.
   ¿Qué combinación de eval de proceso (119) y permisos (116) convierte ese caso en
   fallo visible, y por qué la tasa de éxito ingenua lo premiaba?
2. ¿Por qué el "primer paso divergente" es mejor unidad de diagnóstico que el paso
   donde el agente se rindió?
3. El laboratorio declara "ocho ejemplos no estiman desempeño real". ¿Qué decisión
   podrías tomar igualmente con esos 8 ejemplos y cuál sería irresponsable tomar?